In [1]:
import gymnasium as gym

from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy

In [3]:
# Create environment
env = gym.make("LunarLander-v3", render_mode="rgb_array")

In [4]:
# Instantiate the agent
model = DQN("MlpPolicy", env, verbose=1)
# Train the agent and display a progress bar
model.learn(total_timesteps=int(2e5), progress_bar=True)
# Save the agent
model.save("dqn_lunar")
del model  # delete trained model to demonstrate loading

Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


Output()

----------------------------------
| rollout/            |          |
|    ep_len_mean      | 91       |
|    ep_rew_mean      | -171     |
|    exploration_rate | 0.983    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 643      |
|    time_elapsed     | 0        |
|    total_timesteps  | 364      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 1.6      |
|    n_updates        | 65       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 90.9     |
|    ep_rew_mean      | -188     |
|    exploration_rate | 0.965    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 737      |
|    time_elapsed     | 0        |
|    total_timesteps  | 727      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 4.99     |
|    n_updates      

In [5]:
# Load the trained agent
# NOTE: if you have loading issue, you can pass `print_system_info=True`
# to compare the system on which the model was trained vs the current one
# model = DQN.load("dqn_lunar", env=env, print_system_info=True)
model = DQN.load("dqn_lunar", env=env)

Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


In [222]:
# Evaluate the agent
# NOTE: If you use wrappers with your environment that modify rewards,
#       this will be reflected here. To evaluate with original rewards,
#       wrap environment in a "Monitor" wrapper before other wrappers.
mean_reward, std_reward = evaluate_policy(model, model.get_env(), n_eval_episodes=10)

mean_reward:-200.00 +/- 0.00


In [15]:
# Enjoy trained agent
#vec_env = model.get_env()
#obs = vec_env.reset()
#for i in range(1000):
#    action, _states = model.predict(obs, deterministic=True)
#    obs, rewards, dones, info = vec_env.step(action)
#    vec_env.render("human")



In [8]:
# Set up fake display; otherwise rendering will fail
import os
os.system("Xvfb :1 -screen 0 1024x768x24 &")
os.environ['DISPLAY'] = ':1'

In [9]:
import base64
from pathlib import Path

from IPython import display as ipythondisplay


def show_videos(video_path="", prefix=""):
    """
    Taken from https://github.com/eleurent/highway-env

    :param video_path: (str) Path to the folder containing videos
    :param prefix: (str) Filter the video, showing only the only starting with this prefix
    """
    html = []
    for mp4 in Path(video_path).glob("{}*.mp4".format(prefix)):
        video_b64 = base64.b64encode(mp4.read_bytes())
        html.append(
            """<video alt="{}" autoplay 
                    loop controls style="height: 400px;">
                    <source src="data:video/mp4;base64,{}" type="video/mp4" />
                </video>""".format(
                mp4, video_b64.decode("ascii")
            )
        )
    ipythondisplay.display(ipythondisplay.HTML(data="<br>".join(html)))

In [10]:
from stable_baselines3.common.vec_env import VecVideoRecorder, DummyVecEnv


def record_video(
    env_id,
    model,
    video_length=500,
    prefix="",
    video_folder="videos/",
):
    """
    :param env_id: (str)
    :param model: (RL model)
    :param video_length: (int)
    :param prefix: (str)
    :param video_folder: (str)
    """
    eval_env = DummyVecEnv([lambda: gym.make(env_id, render_mode="rgb_array")])
    # Start the video at step=0 and record 500 steps
    eval_env = VecVideoRecorder(
        eval_env,
        video_folder=video_folder,
        record_video_trigger=lambda step: step == 0,
        video_length=video_length,
        name_prefix=prefix,
    )

    obs = eval_env.reset()
    for _ in range(video_length):
        action, _ = model.predict(obs, deterministic=False)
        obs, _, _, _ = eval_env.step(action)

    # Close the video recorder
    eval_env.close()

In [14]:
record_video("LunarLander-v3", model, video_length=5000, prefix="dqn-lunarlander")

Saving video to C:\Users\masanz\Documents\Reinforcement learning\Material clase 24_25\Codigos RL 2024_2025\8_DQN\videos\dqn-lunarlander-step-0-to-step-5000.mp4
Moviepy - Building video C:\Users\masanz\Documents\Reinforcement learning\Material clase 24_25\Codigos RL 2024_2025\8_DQN\videos\dqn-lunarlander-step-0-to-step-5000.mp4.
Moviepy - Writing video C:\Users\masanz\Documents\Reinforcement learning\Material clase 24_25\Codigos RL 2024_2025\8_DQN\videos\dqn-lunarlander-step-0-to-step-5000.mp4



Moviepy - Done !
Moviepy - video ready C:\Users\masanz\Documents\Reinforcement learning\Material clase 24_25\Codigos RL 2024_2025\8_DQN\videos\dqn-lunarlander-step-0-to-step-5000.mp4


In [17]:
show_videos("videos", prefix="dqn-lunarlander")